# Introducción a la Ciencia de Datos: Tarea 1

Este notebook contiene el código de base para realizar la Tarea 1 del curso. Puede copiarlo en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y la librería Pandas. Si no tiene ninguna familiaridad con la librería, se recomienda realizar algún tutorial introductorio (ver debajo).
También se espera que los alumnos sean proactivos a la hora de consultar las documentaciones de las librerías y del lenguaje, para entender el código provisto.
Además de los recursos provistos en la [página del curso](https://eva.fing.edu.uy/course/view.php?id=1378&section=1), los siguientes recursos le pueden resultar interesantes:
 - [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) y [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): Son parte de la documentación en la página oficial de Pandas.
 - [Kaggle Learn](https://www.kaggle.com/learn): Incluye tutoriales de Python y Pandas.


Si desea utilizar el lenguaje R y está dispuesto a no utilizar (o traducir) este código de base, también puede hacerlo.

En cualquier caso, **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook (ver [README](https://github.com/DonBraulio/introCD)).

In [ ]:
from time import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datasets import load_dataset

# Nuestros paquetes
import sweetviz as swv

## Descarga del dataset
En esta tarea se utilizará una base de datos abierta que contiene artículos de noticias publicados en distintos medios de prensa, con la finalidad de realizar una clasificación de textos según el medio de prensa al que pertenecen. [Link](https://huggingface.co/datasets/rjac/all-the-news-2-1-Component-one?utm_source=chatgpt.com) \
\
Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas. La constante `DATA_PATH` determina la ubicación donde se almacenaran los datos. \
\
El dataset entero pesa ~8.3gb. Para evitar demoras en la descarga/procesamiento vamos a utilizar el parámetro `streaming=True` y hacer un muestreo aleatorio para descargar una porción de los datos lo más representativa posible.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train",cache_dir="../data")
df = ds.to_pandas()

## Lectura de Datos

In [ ]:
# Veamos las primeras filas del DataFrame
df.head()

In [ ]:
# Veamos información general del DataFrame
df.info()

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos
Analice el contenido del DataFrame. Reporte si existen datos faltantes en algún campo, o cualquier otro problema de calidad de datos que encuentre. \
En particular, analice la cantidad de artículos por medio de prensa, y a partir de este punto trabaje con los **cinco medios con mayor cantidad de artículos**.

In [ ]:
# Analice datos faltantes por columna
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Filas faltantes': missing, 'Porcentaje de filas vacias (%)': missing_pct})
missing_df_filtered = missing_df[missing_df['Filas faltantes'] > 0].sort_values('Porcentaje de filas vacias (%)', ascending=False)

print(f"Total de filas: {len(df)}")
print(f"Total de columnas: {len(df.columns)}\n")

if not missing_df_filtered.empty:
    print("Columnas con datos faltantes:")
    display(missing_df_filtered)
else:
    print("No hay datos faltantes en ninguna columna.")

In [ ]:
# Reporte automático de calidad de datos con sweetviz
# Si no está instalado: uv add sweetviz

report = swv.analyze(df)
report.show_notebook()

## Chequeos manuales de calidad

In [ ]:
import re

print("1. Redundancia entre idx y article_idx?")
# Tiene sentido tener esas dos columnas idx y article idx? Parece que son lo mismo?
n_diff = (df['idx'] != df['article_idx']).sum()
print(f"   Filas donde idx != article_idx: {n_diff}")
if n_diff == 0:
    print("Columnas idénticas. Parece haber redundancia en estas dos columnas.")
else:
    display(df[df['idx'] != df['article_idx']][['idx', 'article_idx']].head())

print("-------------------------------")

print("2. Formatos distintos en la columna date?")
# Cuantos formatos distintos de fecha hay en la columna date?
iso_date     = df['date'].str.match(r'^\d{4}-\d{2}-\d{2}$')
iso_datetime = df['date'].str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$')
other_fmt    = ~(iso_date | iso_datetime)
print(f"   YYYY-MM-DD             : {iso_date.sum()}")
print(f"   YYYY-MM-DD HH:MM:SS    : {iso_datetime.sum()}")
print(f"   Otro formato           : {other_fmt.sum()}")
if other_fmt.sum() > 0:
    print("   Ejemplos:")
    display(df.loc[other_fmt, 'date'].head(5))

print("-------------------------------")

print("3. Por qué month está como float?")
print(f"   dtype: {df['month'].dtype}")
print(f"   NaN en month: {df['month'].isnull().sum()}")
print(f"   Muestra: {sorted(df['month'].dropna().unique())[:8]}")

print("-------------------------------")

print("4. Consistencia date vs year / month / day")
date_parsed = pd.to_datetime(df['date'], errors='coerce')
year_num    = pd.to_numeric(df['year'],  errors='coerce')
month_num   = pd.to_numeric(df['month'], errors='coerce')
day_num     = pd.to_numeric(df['day'],   errors='coerce')

inc_year  = (date_parsed.dt.year  != year_num)  & date_parsed.notna() & year_num.notna()
inc_month = (date_parsed.dt.month != month_num) & date_parsed.notna() & month_num.notna()
inc_day   = (date_parsed.dt.day   != day_num)   & date_parsed.notna() & day_num.notna()

print(f"   Año  inconsistente: {inc_year.sum()}")
print(f"   Mes  inconsistente: {inc_month.sum()}")
print(f"   Día  inconsistente: {inc_day.sum()}")
if (inc_year | inc_month | inc_day).any():
    display(df.loc[inc_year | inc_month | inc_day, ['date','year','month','day']].head(5))
print("year/month/day son redundantes con 'date'; se pueden eliminar.")

print("-------------------------------")

print("5. Validación de URLs")
url_re  = re.compile(r'^https?://[^\s/$.?#][^\s]*$')
has_url = df['url'].notna()
valid   = df.loc[has_url, 'url'].apply(lambda x: bool(url_re.match(str(x))))
print(f"   URLs presentes : {has_url.sum()}")
print(f"   URLs válidas   : {valid.sum()} ({valid.mean()*100:.1f}%)")
print(f"   URLs inválidas : {(~valid).sum()}")
if (~valid).sum() > 0:
    display(df.loc[has_url][~valid][['url', 'publication']].head(10))

In [ ]:
# Analice la cantidad de artículos por medio de prensa
articles_per_pub = df['publication'].value_counts()

print(f"Total de medios de prensa: {len(articles_per_pub)}")
print(f"\nDistribución de artículos por medio:")
display(articles_per_pub.to_frame('Artículos'))

# Los 5 medios con más articulos
top_5_publications = articles_per_pub.head(5).index.tolist()
df_top_5 = df[df['publication'].isin(top_5_publications)].copy()

## B. Visualización temporal
Genere una gráfica que permita visualizar los artículos de los cinco medios a lo largo del tiempo, con alguna escala temporal adecuada. \
Comente si se identifican momentos de mayor actividad o patrones temporales en la cobertura.

In [ ]:
# Normalizar columna date: format='mixed' para manejar YYYY-MM-DD y YYYY-MM-DD HH:MM:SS
df_top_5['date_parsed'] = pd.to_datetime(df_top_5['date'], errors='coerce', format='mixed')
df_top_5['date_day']    = df_top_5['date_parsed'].dt.to_period('D')
df_top_5['date_month']  = df_top_5['date_parsed'].dt.to_period('M')
df_top_5['date_year']   = df_top_5['date_parsed'].dt.to_period('Y')

# Ver si alguna es NaT luego de la normalizada
n_unparsed = df_top_5['date_parsed'].isna().sum()
print(f"Fechas no parseadas: {n_unparsed} ({n_unparsed/len(df_top_5)*100:.1f}%)")

def count_by_period(period_col):
    return (
        df_top_5.groupby([period_col, 'publication'])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )

counts_day   = count_by_period('date_day')
counts_month = count_by_period('date_month')
counts_year  = count_by_period('date_year')

fig, axes = plt.subplots(2, 1, figsize=(14, 14))
fig.suptitle('Artículos publicados por los 5 medios a lo largo del tiempo', fontsize=14, y=1.01)

# Por mes
counts_month.plot(ax=axes[0], linewidth=1.2, alpha=0.85, marker='o', markersize=3)
axes[0].set_title('Granularidad: mes')
axes[0].set_xlabel('')
axes[0].set_ylabel('Artículos')
axes[0].legend(title='Medio', bbox_to_anchor=(1.01, 1), loc='upper left')

# Por año
counts_year.plot(ax=axes[1], kind='bar', alpha=0.85, edgecolor='white')
axes[1].set_title('Granularidad: año')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Artículos')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Medio', bbox_to_anchor=(1.01, 1), loc='upper left')


plt.tight_layout()
plt.show()

In [ ]:
# Patrones cíclicos: día de la semana y mes del año
df_top_5['dow']         = df_top_5['date_parsed'].dt.dayofweek   # 0=lunes … 6=domingo
df_top_5['month_of_yr'] = df_top_5['date_parsed'].dt.month

day_names   = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

avg_dow = (
    df_top_5.groupby(['date_day', 'dow', 'publication'])
    .size()
    .reset_index(name='n')
    .groupby(['dow', 'publication'])['n']
    .mean()
    .unstack()
)
avg_dow.index = [day_names[i] for i in avg_dow.index]

avg_month = (
    df_top_5.groupby(['date_month', 'month_of_yr', 'publication'])
    .size()
    .reset_index(name='n')
    .groupby(['month_of_yr', 'publication'])['n']
    .mean()
    .unstack()
)
avg_month.index = [month_names[i-1] for i in avg_month.index]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

avg_dow.plot(ax=axes[0], marker='o', linewidth=1.5)
axes[0].set_title('Promedio de artículos por día de la semana')
axes[0].set_ylabel('Artículos (promedio)')
axes[0].set_xlabel('')
axes[0].legend(title='Medio', bbox_to_anchor=(1.01, 1), loc='upper left')
axes[0].grid(axis='y', alpha=0.3)

avg_month.plot(ax=axes[1], marker='o', linewidth=1.5)
axes[1].set_title('Promedio de artículos por mes del año')
axes[1].set_ylabel('Artículos (promedio)')
axes[1].set_xlabel('')
axes[1].set_xticks(range(len(avg_month)))
axes[1].set_xticklabels(avg_month.index)
axes[1].legend(title='Medio', bbox_to_anchor=(1.01, 1), loc='upper left')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## C. Limpieza de texto y conteo de palabras
Se provee la función `clean_text(...)` que realiza parte de la normalización del texto. \
**Complete la función** agregando signos de puntuación faltantes y cualquier otra normalización que considere oportuna. \
Compruebe el resultado observando el contenido del DataFrame procesado. Comente todas las transformaciones que haya agregado y justifique.

In [ ]:
def clean_text(df, column_name):

    # Eliminar primeras palabras hasta el primer "\n"
    result = df[column_name].str.replace(r"^[^\n]*\n", "", regex=True)

    # Convertir todo a minúsculas
    result = result.str.lower()

    # Eliminar URLs (http/https): en artículos de noticias aparecen links que no aportan
    # información léxica para clasificación por medio de prensa
    result = result.str.replace(r'https?://\S+', ' ', regex=True)

    # Eliminar todos los signos de puntuación y caracteres no alfanuméricos.
    # El loop original solo cubría 5 símbolos ("[", "\n", ",", ":", "?"); con el patrón
    # [^\w\s] se capturan en un solo paso: . ! ; ( ) [ ] { } " ' - _ / | < > # @ % & * + = ^ ~ `
    # Reemplazar por espacio evita que dos palabras queden pegadas al eliminar un guion o barra.
    result = result.str.replace(r'[^\w\s]', ' ', regex=True)

    # Eliminar números aislados (dígitos solos): fechas, cifras y porcentajes no ayudan
    # a discriminar el estilo editorial de cada medio.
    result = result.str.replace(r'\b\d+\b', ' ', regex=True)

    # Normalizar saltos de línea y tabulaciones a espacios
    result = result.str.replace(r'[\n\r\t]', ' ', regex=True)

    # Colapsar múltiples espacios consecutivos en uno solo
    result = result.str.replace(r'\s+', ' ', regex=True)

    # Eliminar espacios al inicio y al final de cada texto
    result = result.str.strip()

    return result

In [ ]:
# Aplicar clean_text sobre la columna "article" y guardar en "CleanText"
df_top_5["CleanText"] = clean_text(df_top_5, "article")

# Verificar resultado: comparar texto original vs. limpio en 3 artículos
for i, row in df_top_5[["article", "CleanText", "publication"]].dropna().head(3).iterrows():
    print(f"=== [{row['publication']}] ===")
    print("ORIGINAL :", repr(row["article"][:200]))
    print("LIMPIO   :", repr(row["CleanText"][:200]))
    print()

## D. Elección de campos de texto
Discuta si conviene trabajar con:
- sólo el cuerpo del artículo,
- sólo el título,
- o una combinación de ambos.

Justifique brevemente su decisión.

Se eligió trabajar con el **cuerpo del artículo** (`article`).

El objetivo del análisis es construir un perfil de palabras que permita distinguir un medio de otro. Para eso, el volumen de texto importa: los títulos tienen en promedio unas 10 palabras, lo que genera perfiles de frecuencia muy ruidosos y dependientes del tema puntual de cada nota. El cuerpo del artículo tiene entre 200 y 900 palabras según el medio, lo que da mucha más señal para caracterizar el estilo editorial real.

Además, el cuerpo captura cosas que el título no: vocabulario técnico, estructura de los párrafos, referencias cruzadas, etc. Combinar ambos campos no aporta demasiado porque el artículo ya contiene toda la información del título.

## E. Pistas que identifican al medio de prensa
Analice si en el texto aparecen pistas que identifiquen de manera directa al medio de prensa (nombres del medio, URLs, firmas, nombres de secciones, plantillas repetidas, etc.). \
En caso de encontrarlas, comente si considera conveniente eliminarlas o reducir su impacto, y justifique su decisión.

In [ ]:
# TODO: Explore el texto buscando pistas que identifiquen directamente al medio de prensa
# Por ejemplo, busque nombres de medios, URLs, firmas, etc.

## F. Restricción por sección o período temporal
Evalúe si conviene restringir el análisis a artículos de una misma sección temática o de un período temporal acotado, con el objetivo de reducir el efecto del tema sobre una futura tarea de clasificación por medio. \
No es necesario implementar esta restricción, pero sí discutir sus posibles ventajas y desventajas.

*TODO: Escriba su análisis en el informe.*

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio
Realice una visualización que permita comparar las palabras más frecuentes de cada uno de los cinco medios de prensa. \
Sin necesidad de implementarlo, proponga ideas para modificar esta visualización con el fin de encontrar diferencias entre secciones temáticas, fechas, o tipos de noticias.

In [ ]:
# TODO: Realice una visualización que permita comparar las palabras más frecuentes
# de cada uno de los cinco medios de prensa.
# - ¿Encuentra algún problema en los resultados?


## B. Medios con mayor cantidad de palabras
Corra el código que permite encontrar los medios con mayor cantidad de palabras. \
En caso de encontrar algún problema luego de realizar la visualización, comente a qué se debe y proponga formas de resolverlo.

In [ ]:
# TODO: Busque los medios con mayor cantidad de palabras

## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

In [ ]:
# TODO: Construya una matriz de 5x5, donde cada fila y columna corresponden a un medio de prensa,
# y la entrada (i,j) contiene la cantidad de veces que el medio "i" menciona al medio "j".

# mentions_matrix = ...


In [ ]:
# Opcional: Genere un grafo dirigido con la matriz de adyacencia para visualizar las menciones.
# Puede ser útil la biblioteca networkx.



## D. Preguntas propuestas
Proponga al menos tres preguntas que se podrían intentar responder a partir de estos datos, y mencione posibles caminos para responderlas, sin implementar nada.

*TODO: Escriba sus preguntas y posibles caminos en el informe.*